In [ ]:
!pip uninstall torchvision -y
!pip install whisperx
!pip install pyngrok
!pip install speechbrain

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import gc
import time
import tempfile
from typing import Optional
from collections import deque

import numpy as np
import torch
import uvicorn
import whisperx
from fastapi import FastAPI, UploadFile, File, Header, HTTPException, Form
from whisperx.diarize import DiarizationPipeline
from transformers import AutoFeatureExtractor, WavLMForXVector
from google.colab import userdata
from pyngrok import ngrok

app = FastAPI()

# --- Настройки ---
API_KEY  = userdata.get("API_KEY")
HF_TOKEN = userdata.get("HF_TOKEN")
DEVICE   = "cuda"
MAX_FILE_SIZE_MB = 50

# ---------------------------------------------------------------------------
# Пороги сходства
# ---------------------------------------------------------------------------
THRESHOLD_CONFIDENT_MATCH = 0.75   # ≥ 0.75 → уверенное совпадение, обновляем центроид
THRESHOLD_SOFT_MATCH      = 0.60   # 0.60–0.75 → серая зона: назначаем, но не обновляем
THRESHOLD_FORCE_NEW       = 0.45   # < 0.45 → явно чужой голос, новый профиль
SIMILARITY_UPDATE_MIN     = 0.65   # минимум для обновления центроида

# Порог для склейки нового куска с существующей pending-записью.
# Намеренно ниже основных — короткие эмбеддинги ненадёжны.
PENDING_MATCH_THRESHOLD   = 0.55

# ---------------------------------------------------------------------------
# Управление реестром
# ---------------------------------------------------------------------------
CALIBRATION_WINDOW = 300.0  # секунд — окно калибровки с начала сессии
MAX_EXTRA_SLOTS    = 2      # доп. слоты сверх лимита для опоздавших

# При 20s чанках фразы короче — 1.0s достаточно для надёжного эмбеддинга
MIN_SPEAKER_DURATION = 1.0  # секунд

# При 20s чанках 3 чанка = 60s ожидания — разумный таймаут для призраков
PENDING_MAX_CHUNKS = 3

EMBEDDING_BUFFER_SIZE = 10


# ---------------------------------------------------------------------------
# Профиль спикера
# ---------------------------------------------------------------------------
class SpeakerProfile:
    def __init__(self, speaker_id: str, initial_embedding: np.ndarray):
        self.id       = speaker_id
        self.centroid = initial_embedding.copy()
        self.buffer: deque = deque([initial_embedding.copy()], maxlen=EMBEDDING_BUFFER_SIZE)
        self.segment_count = 0

    @property
    def display_name(self) -> str:
        return self.id

    def vote_similarity(self, embedding: np.ndarray) -> float:
        """Среднее сходство по буферу — устойчивее одного центроида."""
        if not self.buffer:
            return -1.0
        sims = [
            float(np.dot(e, embedding) / (np.linalg.norm(e) * np.linalg.norm(embedding) + 1e-8))
            for e in self.buffer
        ]
        if not sims:
            return -1.0
        return float(np.mean(sims))

    def update(self, embedding: np.ndarray, sim: float):
        """Обновляем центроид только при уверенном совпадении."""
        self.buffer.append(embedding.copy())
        self.segment_count += 1
        if sim >= SIMILARITY_UPDATE_MIN:
            alpha = 0.05 + 0.10 * (sim - SIMILARITY_UPDATE_MIN)
            self.centroid = (1 - alpha) * self.centroid + alpha * embedding

    def soft_assign(self, embedding: np.ndarray):
        """Серая зона: в буфер, но без обновления центроида."""
        self.buffer.append(embedding.copy())
        self.segment_count += 1


# ---------------------------------------------------------------------------
# Pending-запись для коротких фраз
# ---------------------------------------------------------------------------
class PendingEntry:
    """
    Накапливает короткие аудиофрагменты одного спикера across чанков.
    Когда total_duration >= MIN_SPEAKER_DURATION → flush → регистрация.
    Если не пополнялась PENDING_MAX_CHUNKS чанков → сброс (призрак).
    """
    def __init__(self, chunks: list, duration: float,
                 rough_emb: Optional[np.ndarray]):
        self.chunks:         list                  = list(chunks)
        self.total_duration: float                = duration
        self.rough_emb:      Optional[np.ndarray] = rough_emb
        self.chunks_seen:    int                  = 1

    def add(self, chunks: list, duration: float,
            rough_emb: Optional[np.ndarray]):
        self.chunks.extend(chunks)
        self.total_duration += duration
        self.chunks_seen    += 1
        if rough_emb is not None:
            self.rough_emb = rough_emb

    def tick(self):
        """Вызывается когда спикер НЕ появился в очередном чанке."""
        self.chunks_seen += 1

    def is_ready(self) -> bool:
        return self.total_duration >= MIN_SPEAKER_DURATION

    def is_stale(self) -> bool:
        return self.chunks_seen >= PENDING_MAX_CHUNKS

    def get_combined(self) -> np.ndarray:
        return peak_normalize(np.concatenate(self.chunks))


# ---------------------------------------------------------------------------
# Глобальное состояние сессии
# ---------------------------------------------------------------------------
profiles:             dict[str, SpeakerProfile] = {}
speaker_counter:      int                       = 0
session_max_speakers: Optional[int]             = None
session_start_time:   float                     = time.time()
pending_pool:         list                      = []  # list[PendingEntry]


def _is_calibration_phase() -> bool:
    elapsed = time.time() - session_start_time
    if elapsed < CALIBRATION_WINDOW:
        return True
    if session_max_speakers is None:
        return True
    return len(profiles) < session_max_speakers


def _extra_slots_available() -> bool:
    if session_max_speakers is None:
        return True
    return len(profiles) < session_max_speakers + MAX_EXTRA_SLOTS


def _register_new(embedding: np.ndarray) -> SpeakerProfile:
    global speaker_counter
    speaker_counter += 1
    new_id  = f"Игрок_{speaker_counter}"
    profile = SpeakerProfile(new_id, embedding)
    profiles[new_id] = profile
    return profile


# ---------------------------------------------------------------------------
# Загрузка моделей
# ---------------------------------------------------------------------------
try:
    model = whisperx.load_model(
        "large-v2",
        DEVICE,
        compute_type="float16",
        vad_options={"vad_onset": 0.5, "vad_offset": 0.363}
    )
    diarize_model = DiarizationPipeline(token=HF_TOKEN, device=DEVICE)

    feature_extractor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
    wavlm_model = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(DEVICE)
    wavlm_model.eval()
    print(">>> Модели успешно загружены.")
except Exception as e:
    raise RuntimeError(f"Ошибка загрузки моделей: {e}")


# ---------------------------------------------------------------------------
# Вспомогательные функции
# ---------------------------------------------------------------------------
def check_api_key(x_api_key: str):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")


def peak_normalize(chunk: np.ndarray) -> np.ndarray:
    peak = np.max(np.abs(chunk))
    if peak < 1e-8:
        return chunk
    return chunk / peak


def _cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def _run_wavlm(combined: np.ndarray, sample_rate: int) -> Optional[np.ndarray]:
    """Извлекает эмбеддинг из уже нормализованного аудио."""
    try:
        inputs = feature_extractor(
            combined, sampling_rate=sample_rate, return_tensors="pt", padding=True
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            emb = wavlm_model(**inputs).embeddings
        return emb.squeeze().cpu().numpy()
    except Exception as e:
        print(f">>> Ошибка WavLM: {e}")
        return None


# ---------------------------------------------------------------------------
# Pending-буфер: добавление и сброс
# ---------------------------------------------------------------------------
def add_to_pending(chunks: list, duration: float,
                   rough_emb: Optional[np.ndarray]):
    """
    Ищет подходящую pending-запись по rough_emb.
    Если похожей нет — создаёт новую.
    """
    global pending_pool

    if rough_emb is not None:
        best_entry, best_sim = None, -1.0
        for entry in pending_pool:
            if entry.rough_emb is None:
                continue
            sim = _cosine_sim(rough_emb, entry.rough_emb)
            if sim > best_sim:
                best_sim, best_entry = sim, entry

        if best_sim >= PENDING_MATCH_THRESHOLD and best_entry is not None:
            best_entry.add(chunks, duration, rough_emb)
            print(f">>> pending: добавлено к записи "
                  f"(sim={best_sim:.2f}, итого {best_entry.total_duration:.1f}s)")
            return

    pending_pool.append(PendingEntry(chunks, duration, rough_emb))
    print(f">>> pending: новая запись ({duration:.1f}s, пул: {len(pending_pool)})")


def flush_pending(sample_rate: int):
    """
    Вызывается в начале каждого чанка.
    Регистрирует готовые записи, удаляет устаревшие.
    """
    global pending_pool

    remaining = []
    for entry in pending_pool:
        if entry.is_ready():
            emb = _run_wavlm(entry.get_combined(), sample_rate)
            if emb is not None:
                profile = match_or_register(emb)
                print(f">>> pending → {profile.display_name} "
                      f"({entry.total_duration:.1f}s, {entry.chunks_seen} чанков)")
            else:
                print(f">>> pending: WavLM не смог извлечь эмбеддинг, сброс")
        elif entry.is_stale():
            print(f">>> pending: устарело "
                  f"({entry.chunks_seen} чанков, {entry.total_duration:.1f}s) — сброс")
        else:
            entry.tick()
            remaining.append(entry)

    pending_pool = remaining


# ---------------------------------------------------------------------------
# Сопоставление эмбеддинга с реестром
# ---------------------------------------------------------------------------
def match_or_register(embedding: np.ndarray) -> SpeakerProfile:
    """
    Четыре зоны решений без пересечений:

    ≥ 0.75          уверенное совпадение → обновляем центроид
    0.60 – 0.75     серая зона → назначаем, не обновляем
    0.45 – 0.60     слабое → новый (калибровка) / назначаем (боевой)
    < 0.45          явно чужой → новый даже при полном реестре
    """
    best_profile, best_sim = None, -1.0
    for profile in profiles.values():
        sim = profile.vote_similarity(embedding)
        if sim > best_sim:
            best_sim, best_profile = sim, profile

    calibration = _is_calibration_phase()

    # ── Зона 1: уверенное совпадение ─────────────────────────────────────────
    if best_sim >= THRESHOLD_CONFIDENT_MATCH and best_profile is not None:
        best_profile.update(embedding, best_sim)
        print(f">>> [{best_sim:.2f}] уверенное → {best_profile.display_name}")
        return best_profile

    # ── Зона 2: серая зона (0.60 – 0.75) ─────────────────────────────────────
    if THRESHOLD_SOFT_MATCH <= best_sim < THRESHOLD_CONFIDENT_MATCH:
        if calibration:
            profile = _register_new(embedding)
            print(f">>> [{best_sim:.2f}] серая + калибровка → новый {profile.display_name}")
            return profile
        else:
            best_profile.soft_assign(embedding)
            print(f">>> [{best_sim:.2f}] серая → {best_profile.display_name} (без обновления)")
            return best_profile

    # ── Зона 3: явно чужой (< 0.45) ──────────────────────────────────────────
    if best_sim < THRESHOLD_FORCE_NEW:
        if _extra_slots_available():
            profile = _register_new(embedding)
            print(f">>> [{best_sim:.2f}] чужой → новый {profile.display_name}")
            return profile
        else:
            best_profile.soft_assign(embedding)
            print(f">>> [{best_sim:.2f}] чужой, слоты исчерпаны → {best_profile.display_name}")
            return best_profile

    # ── Зона 4: слабое совпадение (0.45 – 0.60) ──────────────────────────────
    if calibration:
        profile = _register_new(embedding)
        print(f">>> [{best_sim:.2f}] слабое + калибровка → новый {profile.display_name}")
        return profile
    else:
        best_profile.soft_assign(embedding)
        print(f">>> [{best_sim:.2f}] слабое → {best_profile.display_name} (без обновления)")
        return best_profile


# ---------------------------------------------------------------------------
# Основной маппинг спикеров
# ---------------------------------------------------------------------------
def remap_speakers(segments: list, audio: np.ndarray, sample_rate: int = 16000) -> list:
    """
    Заменяет временные SPEAKER_XX диаризатора на стабильные ID реестра.

    Ключевые принципы:
    1. Каждый сегмент обрабатывается независимо — эмбеддинг на каждую фразу.
    2. Профиль хранится по ИНДЕКСУ сегмента (seg_to_profile[idx]),
       а не по SPEAKER_XX. Это исправляет баг когда pyannote помечает
       разные голоса одним SPEAKER_XX — каждая фраза получает свой профиль.
    3. Короткие фразы (< MIN_SPEAKER_DURATION) → pending_pool для накопления.
    """
    flush_pending(sample_rate)

    stats = {"direct": 0, "pending": 0, "skipped": 0}

    # Профиль хранится по индексу сегмента — не по SPEAKER_XX
    # Это ключевое исправление: если SPEAKER_00 содержит фразы разных людей,
    # каждая фраза получает свой независимый профиль
    seg_to_profile: dict[int, SpeakerProfile] = {}

    for idx, seg in enumerate(segments):
        spk = seg.get("speaker", "UNKNOWN")
        if spk == "UNKNOWN":
            continue

        dur = seg["end"] - seg["start"]

        if dur < 0.3:
            stats["skipped"] += 1
            continue

        chunk = audio[int(seg["start"] * sample_rate):int(seg["end"] * sample_rate)]
        chunk = peak_normalize(chunk)

        if dur >= MIN_SPEAKER_DURATION:
            # ── Фраза достаточно длинная → сразу в реестр ────────────────────
            emb = _run_wavlm(chunk, sample_rate)
            if emb is None:
                continue
            profile = match_or_register(emb)
            seg_to_profile[idx] = profile
            stats["direct"] += 1

        else:
            # ── Короткая фраза → pending ──────────────────────────────────────
            rough_emb = _run_wavlm(chunk, sample_rate)
            add_to_pending([chunk], dur, rough_emb)
            stats["pending"] += 1

    print(f">>> фраз: реестр={stats['direct']}, "
          f"pending={stats['pending']}, пропущено={stats['skipped']}")

    # Назначаем каждому сегменту его собственный профиль по индексу
    for idx, seg in enumerate(segments):
        if idx in seg_to_profile:
            seg["speaker"] = seg_to_profile[idx].display_name

    return segments


# ---------------------------------------------------------------------------
# Эндпоинты
# ---------------------------------------------------------------------------
@app.post("/reset")
async def reset(x_api_key: str = Header(...)):
    check_api_key(x_api_key)
    global profiles, speaker_counter, session_max_speakers, \
           session_start_time, pending_pool
    profiles             = {}
    speaker_counter      = 0
    session_max_speakers = None
    session_start_time   = time.time()
    pending_pool         = []
    gc.collect()
    torch.cuda.empty_cache()
    print(">>> Память и реестр очищены. Новая игра. Калибровка началась.")
    return {"status": "memory cleared"}


@app.post("/process_chunk")
async def process_chunk(
    file:         UploadFile = File(...),
    num_speakers: int        = Form(None),
    min_speakers: int        = Form(None),
    max_speakers: int        = Form(None),
    x_api_key:    str        = Header(...)
):
    check_api_key(x_api_key)

    contents = await file.read()
    if len(contents) > MAX_FILE_SIZE_MB * 1024 * 1024:
        raise HTTPException(status_code=413, detail="File too large")

    with tempfile.NamedTemporaryFile(dir="/dev/shm", suffix=".wav", delete=False) as tmp:
        tmp.write(contents)
        temp_path = tmp.name

    try:
        audio = whisperx.load_audio(temp_path)

        if len(audio) < 32000:
            print(f">>> Чанк слишком короткий ({len(audio)} сэмплов), пропускаем.")
            return {"segments": []}

        # Транскрипция
        result = model.transcribe(audio, batch_size=16, language="ru")

        # Выравнивание
        model_a, metadata = whisperx.load_align_model(
            language_code=result["language"], device=DEVICE
        )
        result = whisperx.align(result["segments"], model_a, metadata, audio, DEVICE)
        del model_a
        gc.collect()
        torch.cuda.empty_cache()

        # Запоминаем лимит реестра (только один раз за сессию)
        global session_max_speakers
        if session_max_speakers is None:
            if num_speakers is not None:
                session_max_speakers = num_speakers
            elif max_speakers is not None:
                session_max_speakers = max_speakers
            if session_max_speakers is not None:
                print(f">>> Лимит реестра: {session_max_speakers} "
                      f"(+ до {MAX_EXTRA_SLOTS} доп. слотов)")

        # Диаризация
        diarize_kwargs = {}
        if min_speakers is not None:
            diarize_kwargs["min_speakers"] = min_speakers
        if max_speakers is not None:
            diarize_kwargs["max_speakers"] = max_speakers

        # подсказываем pyannote сколько голосов искать
        if "max_speakers" not in diarize_kwargs and session_max_speakers is not None:
            diarize_kwargs["max_speakers"] = session_max_speakers

        try:
            diarize_segments = diarize_model(audio, **diarize_kwargs)
        except Exception as e:
            if "n_samples" in str(e) and "n_clusters" in str(e):
                print(">>> Мало сегментов, снимаем min_speakers")
                try:
                    fallback = {"max_speakers": diarize_kwargs["max_speakers"]} \
                               if "max_speakers" in diarize_kwargs else {}
                    diarize_segments = diarize_model(audio, **fallback)
                except Exception as e2:
                    print(f">>> Диаризация не удалась: {e2}")
                    return {"segments": result["segments"]}
            else:
                print(f">>> Ошибка диаризации: {e}")
                return {"segments": result["segments"]}

        result = whisperx.assign_word_speakers(diarize_segments, result)

        # ── Диагностика: что нашёл pyannote ──────────────────────────────────
        raw_speakers: dict = {}
        for seg in result["segments"]:
            spk = seg.get("speaker", "UNKNOWN")
            raw_speakers[spk] = raw_speakers.get(spk, 0.0) + seg["end"] - seg["start"]
        print(f">>> pyannote: {len(raw_speakers)} спикеров в чанке:")
        for spk, dur in sorted(raw_speakers.items()):
            flag = " ← pending" if dur < MIN_SPEAKER_DURATION else ""
            print(f"    {spk}: {dur:.1f}s{flag}")
        # ─────────────────────────────────────────────────────────────────────

        result["segments"] = remap_speakers(result["segments"], audio)

        elapsed = time.time() - session_start_time
        phase   = "калибровка" if _is_calibration_phase() else "боевой режим"
        print(f">>> [{elapsed:.0f}s] {phase} | "
              f"реестр: {len(profiles)} | pending: {len(pending_pool)}")

    except Exception as e:
        print(f">>> Ошибка обработки чанка: {e}")
        torch.cuda.empty_cache()
        return {"segments": []}

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        torch.cuda.empty_cache()

    return {"segments": result["segments"]}


# --- Запуск ---
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
public_url = ngrok.connect(8000)
print(f">>> Публичный URL: {public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()